In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tbparse

In [ ]:
final_csv_path = './result_csvs_and_pdfs/FINAL.csv'
final_df = pd.read_csv(final_csv_path)
final_df

In [ ]:
final_df['label'] = final_df[['model','train_data','test_data']].apply(lambda x:'\n'.join(x), axis=1)
final_df['percent_epochs'] = final_df['train_epochs']/final_df['max_epochs']
final_df.columns

In [ ]:
# get dataframe from lightning log

def get_df_from_log(log_path):
    reader = tbparse.SummaryReader(log_path)
    return reader.scalars

In [ ]:

# get dfs from logs
hyperparam_dfs = {}
train_dfs = {}
test_dfs = {}
summary_dfs = {}

for index,row in final_df.iterrows():
    h_df = get_df_from_log(row.hyperparam_log_path)
    hyperparam_dfs[(row.model,row.train_data,row.test_data)] = h_df
    train_df = get_df_from_log(row.train_log_path)
    train_dfs[(row.model,row.train_data,row.test_data)] = train_df
    test_df = get_df_from_log(row.test_log_path)
    test_dfs[(row.model,row.train_data,row.test_data)] = test_df
    summary_df = get_df_from_log(row.summary_log_path)
    summary_dfs[(row.model,row.train_data,row.test_data)] = summary_df


In [ ]:
models = ['model', 'train_data', 'test_data']
hyperparams = ['learning_rate', 'batch_size', 'accum_grad_batches', 'max_epochs', 'feature_length', 'dim_mlp_layers']

melt_df = pd.melt(final_df, id_vars='label', var_name='Metric', value_name='Value')
melt_df = melt_df[melt_df.Metric.isin(hyperparams)]
melt_df

In [ ]:
# barplot hyperparameters combined

df = melt_df.rename(columns={'label': 'Model'})

plt.figure(figsize=(2 * len(hyperparams),8))
plt.grid()
sns.barplot(x='Metric', y='Value', hue='Model', data=df, palette='deep', edgecolor='black')
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.3), ncol=2)
plt.xticks(rotation=45)
plt.tight_layout()
plt.title('Hyper Parameters by Model')
plt.show()

In [ ]:
# individual bar plots

x_measures = {
  "Training Runtime (in secs)": "train_walltime",
  "Test AUROC": "test_auroc",
  "Test Loss": "test_loss",
  "Learning Rate": "learning_rate",
  "Batch Size": "batch_size",
  "Gradient Accumulation": "accum_grad_batches",
  "Max Epochs": "max_epochs",
  "Train Epochs": "train_epochs",
  "Train Steps": "train_steps",
  "Percent of Max Epochs Used": "percent_epochs",
}

for x_title,x_col in x_measures.items():
    sns.barplot(x="label", y=f"{x_col}", data=final_df, hue="label", palette='deep')
    plt.xlabel("(Model,Training Data,Test Data)")
    plt.ylabel(f"{x_title}")
    plt.title(f"{x_title} by Model")
    plt.xticks(rotation=45)
    plt.savefig(f'result_csvs_and_pdfs/{x_col}.barplot.pdf')
    plt.show()

In [ ]:
# line plots over epochs (using summary writer logs)

x_measures = {
  "Runtime over Steps": "walltime_per_batch",
  "Loss over Steps": "loss_per_batch"
}

for x_title,x_col in x_measures.items():
    for key,df in summary_dfs.items():
        df_tag = df[df.tag == x_col]
        plt.plot(df_tag.step, df_tag.value, label=key)

    plt.xlabel("Steps")
    plt.ylabel(f"{x_title}")
    plt.title(f"{x_title} by Model")
    plt.xticks(rotation=45)

    plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
    plt.grid()
    plt.savefig(f'result_csvs_and_pdfs/{x_col}.lineplot.pdf')
    plt.show()

In [ ]:
for key,df in summary_dfs.items():
    df_time = df[df.tag == 'walltime_per_batch']
    df_loss = df[df.tag == 'loss_per_batch']
    plt.plot(df_time.value, df_loss.value, label=key)

x_title = "Training Loss over Time"
plt.xlabel("Walltime (in Seconds)")
plt.ylabel(f"{x_title}")
plt.title(f"{x_title} by Model")
plt.xticks(rotation=45)

plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
plt.grid()
plt.savefig(f'result_csvs_and_pdfs/loss_per_walltime.lineplot.pdf')
plt.show()

In [ ]:
# line plots over epochs (using lightning logs)

x_measures = {
  "Training Loss over Epochs": "train_loss_epoch",
  "Training Loss over Steps": "train_loss_step",
  "Positive Prediction Average over Epochs": "pos_prediction_avg",
  "Negative Prediction Average over Epochs": "neg_prediction_avg",
}

for x_title,x_col in x_measures.items():
    for key,df in train_dfs.items():
        df_tag = df[df.tag == x_col]
        plt.plot(df_tag.step, df_tag.value, label=key)

    plt.xlabel("Epochs")
    plt.ylabel(f"{x_title}")
    plt.title(f"{x_title} by Model")
    plt.xticks(rotation=45)

    plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
    plt.grid()
    plt.savefig(f'result_csvs_and_pdfs/{x_col}.lineplot.pdf')
    plt.show()